# Homework 11: Bootstrap, CLT, and Normal Approximation


7. In `./data/ames_prices.csv`, create an indicator that equals 1 when `Central.Air == 'Y'` and 0 otherwise.
   
   a. Estimate the proportion of homes with central air.
   
   b. Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of this proportion. Report the seed and number of bootstrap replications.
   
   c. Compute the bootstrap standard error as the sample standard deviation of the bootstrap estimates (using `ddof=1` ensures that there's the $1/(N-1)$ degrees of freedom correction).
   
   d. Using the original estimate and the bootstrap standard error, construct Normal 90%, 95%, and 99% confidence intervals.
   
   e. Use each confidence interval to evaluate $H_0:p=.90$. At the matching significance levels $\alpha=.10,.05,.01$, state whether you reject or fail to reject the null hypothesis. Briefly interpret your results.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
ames_prices = pd.read_csv("./data/ames_prices.csv")
ames_prices.head()

,Order,PID,area,price,MS.SubClass,MS.Zoning,Lot.Frontage,Lot.Area,Street,Alley,...,Screen.Porch,Pool.Area,Pool.QC,Fence,Misc.Feature,Misc.Val,Mo.Sold,Yr.Sold,Sale.Type,Sale.Condition
0,1,526301100,1656,215000,20,RL,141.0,31770,Pave,NaN,...,0,0,NaN,NaN,NaN,0,5,2010,WD,Normal
1,2,526350040,896,105000,20,RH,80.0,11622,Pave,NaN,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
2,3,526351010,1329,172000,20,RL,81.0,14267,Pave,NaN,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
3,4,526353030,2110,244000,20,RL,93.0,11160,Pave,NaN,...,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal
4,5,527105010,1629,189900,60,RL,74.0,13830,Pave,NaN,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal


In [3]:
ames_prices.columns

Index(['Order', 'PID', 'area', 'price', 'MS.SubClass', 'MS.Zoning',
       'Lot.Frontage', 'Lot.Area', 'Street', 'Alley', 'Lot.Shape',
       'Land.Contour', 'Utilities', 'Lot.Config', 'Land.Slope', 'Neighborhood',
       'Condition.1', 'Condition.2', 'Bldg.Type', 'House.Style',
       'Overall.Qual', 'Overall.Cond', 'Year.Built', 'Year.Remod.Add',
       'Roof.Style', 'Roof.Matl', 'Exterior.1st', 'Exterior.2nd',
       'Mas.Vnr.Type', 'Mas.Vnr.Area', 'Exter.Qual', 'Exter.Cond',
       'Foundation', 'Bsmt.Qual', 'Bsmt.Cond', 'Bsmt.Exposure',
       'BsmtFin.Type.1', 'BsmtFin.SF.1', 'BsmtFin.Type.2', 'BsmtFin.SF.2',
       'Bsmt.Unf.SF', 'Total.Bsmt.SF', 'Heating', 'Heating.QC', 'Central.Air',
       'Electrical', 'X1st.Flr.SF', 'X2nd.Flr.SF', 'Low.Qual.Fin.SF',
       'Bsmt.Full.Bath', 'Bsmt.Half.Bath', 'Full.Bath', 'Half.Bath',
       'Bedroom.AbvGr', 'Kitchen.AbvGr', 'Kitchen.Qual', 'TotRms.AbvGrd',
       'Functional', 'Fireplaces', 'Fireplace.Qu', 'Garage.Type',
       'Garage.Yr.B

a. Estimated proportion of homes with central air is about 0.93. 

In [4]:
ames_prices['Central.Air'] = ames_prices['Central.Air'].map({'Y': 1, 'N': 0})
prop = ames_prices['Central.Air'].mean()

   b. Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of this proportion. Report the seed and number of bootstrap replications.

In [5]:
seed = 2134
rng = np.random.default_rng(seed)
R = 2_500

replications = []
for r in range(R):
    df_r = ames_prices.sample( frac = 1.0, replace = True, random_state = rng )
    p_r = df_r['Central.Air'].mean()
    replications.append(p_r)
replications = np.array(replications)

In [6]:
print(f'Seed = {seed}')
print(f'Replications = {R}')

Seed = 2134
Replications = 2500


 c. Compute the bootstrap standard error as the sample standard deviation of the bootstrap estimates (using `ddof=1` ensures that there's the $1/(N-1)$ degrees of freedom correction).

In [7]:
se = np.std(replications, ddof=1)
print(f'Bootstrapped SE = {se}')

Bootstrapped SE = 0.00464297935101086


   d. Using the original estimate and the bootstrap standard error, construct Normal 90%, 95%, and 99% confidence intervals.

In [8]:
ci90 = (prop - 1.645*se, prop + 1.645*se)
ci95 = (prop - 1.960*se, prop + 1.960*se)
ci99 = (prop - 2.576*se, prop + 2.576*se)

print(ci90)
print(ci95)
print(ci99)

(np.float64(0.9254681010153687), np.float64(0.9407435030801945))
(np.float64(0.9240055625198003), np.float64(0.9422060415757629))
(np.float64(0.9211454872395776), np.float64(0.9450661168559856))


   e. Use each confidence interval to evaluate $H_0:p=.90$. At the matching significance levels $\alpha=.10,.05,.01$, state whether you reject or fail to reject the null hypothesis. Briefly interpret your results.

- 0.90 is not included in any of the three confidence intervals (90/95/99%), so we can reject $H_0:p=.90$ at the significance levels $\alpha=.10,.05,.01$. 
- There is evidence to suggest that the difference between the houses with central air (around 93%) is significant compared to 90%.

8. In `./data/ames_prices.csv`, estimate the Pearson correlation between `np.arcsinh(area)` and `np.arcsinh(price)`.
   
   a. Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of the correlation by resampling paired rows. Report the seed and number of bootstrap replications.

In [9]:
ames_prices['area'] = np.arcsinh(ames_prices['area'])
ames_prices['price'] = np.arcsinh(ames_prices['price'])

corr = ames_prices['area'].corr(ames_prices['price'])
print(f'Pearson correlation estimate: {corr}')

Pearson correlation estimate: 0.7231661530091354


In [10]:
R = 2_500
rng = np.random.default_rng(seed)

correlations = []
for r in range(R):
    df_r = ames_prices.sample( frac = 1.0, replace = True, random_state = rng )
    corr_r = df_r['area'].corr(df_r['price'])
    correlations.append(corr_r)
correlations = np.array(correlations)

In [11]:
print(f'Seed = {seed}')
print(f'Replications = {R}')

Seed = 2134
Replications = 2500


b. Compute the bootstrap standard error as the sample standard deviation of the bootstrap estimates (using `ddof=1` ensures that there's the $1/(N-1)$ degrees of freedom correction).

In [12]:
se = np.std(correlations, ddof=1)
print(f'Bootstrapped SE = {se}')

Bootstrapped SE = 0.010301378228988963


   c. Using the original estimate and the bootstrap standard error, construct centered-Normal 90%, 95%, and 99% confidence intervals.

In [13]:
ci90 = (corr - 1.645*se, corr + 1.645*se)
ci95 = (corr - 1.960*se, corr + 1.960*se)
ci99 = (corr - 2.576*se, corr + 2.576*se)

print(ci90)
print(ci95)
print(ci99)

(np.float64(0.7062203858224486), np.float64(0.7401119201958223))
(np.float64(0.702975451680317), np.float64(0.7433568543379538))
(np.float64(0.6966298026912598), np.float64(0.749702503327011))


   d. Use each confidence interval to evaluate $H_0:\rho=0$. At the matching significance levels $\alpha=.10,.05,.01$, state whether you reject or fail to reject the null hypothesis. Briefly interpret the association without making a causal claim.

- 0 lies outside of all three of the confidence intervals (90/95/99%), so we have evidence to reject $H_0:\rho=0$ at the significance levels $\alpha=.10,.05,.01$. Larger house size (in area) is highly associated with a higher house price.

   e. Briefly explain why `area` and `price` must remain paired during resampling.

- Area and price must remain paired during resampling because the two are specifically related to each other. Each house area belongs to a specific house price. So, if they weren't paired during resampling, this wouldn't be accounted for and thus could lead to an incorrect sampling distrbution constructed from the bootstrap.

9. In `./data/metabric.csv`, use `Overall Survival (Months)` to estimate the mean recorded time to death or last follow-up. Include both living and deceased patients.
   
a. Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of the mean. Report the seed and number of bootstrap replications.

In [14]:
metabric = pd.read_csv("./data/metabric.csv")
metabric.head()

,Age at Diagnosis,Type of Breast Surgery,Cancer Type,Chemotherapy,Hormone Therapy,Lymph nodes examined positive,Mutation Count,Nottingham prognostic index,Overall Survival (Months),Overall Survival Status,Radio Therapy,TMB (nonsynonymous),Tumor Size,Tumor Stage
0,43.19,BREAST CONSERVING,Breast Cancer,NO,YES,0.0,2.0,4.020,84.633333,0:LIVING,YES,2.615035,10.0,1.0
1,48.87,MASTECTOMY,Breast Cancer,YES,YES,1.0,2.0,4.030,163.700000,1:DECEASED,NO,2.615035,15.0,2.0
2,47.68,MASTECTOMY,Breast Cancer,YES,YES,3.0,1.0,4.050,164.933333,0:LIVING,YES,1.307518,25.0,2.0
3,76.97,MASTECTOMY,Breast Cancer,YES,YES,8.0,2.0,6.080,41.366667,1:DECEASED,YES,2.615035,40.0,2.0
4,78.77,MASTECTOMY,Breast Cancer,NO,YES,0.0,4.0,4.062,7.800000,1:DECEASED,YES,5.230071,31.0,4.0


In [15]:
mean_survival_months = metabric['Overall Survival (Months)'].mean()
mean_survival_months

np.float64(129.09429138744105)

In [16]:
seed = 2134
rng = np.random.default_rng(seed)
R = 2_500

survival = []
for r in range(R):
    df_r = metabric.sample( frac = 1.0, replace = True, random_state = rng )
    m_r = df_r['Overall Survival (Months)'].mean()
    survival.append(m_r)
survival = np.array(survival)

In [17]:
print(f'Seed = {seed}')
print(f'Replications = {R}')

Seed = 2134
Replications = 2500


b. Compute the bootstrap standard error as the sample standard deviation of the bootstrap estimates (using `ddof=1` ensures that there's the $1/(N-1)$ degrees of freedom correction).

In [18]:
metabric_se = np.std(survival, ddof=1)
print(f'Bootstrapped SE = {metabric_se}')

Bootstrapped SE = 2.0633807631616405


c. Using the original estimate and the bootstrap standard error, construct centered-Normal 90%, 95%, and 99% confidence intervals.

In [19]:
ci90 = (mean_survival_months - 1.645*metabric_se, mean_survival_months + 1.645*metabric_se)
ci95 = (mean_survival_months - 1.960*metabric_se, mean_survival_months + 1.960*metabric_se)
ci99 = (mean_survival_months - 2.576*metabric_se, mean_survival_months + 2.576*metabric_se)

print(f'90% Confidence Intervel: {ci90}')
print(f'95% Confidence Intervel: {ci95}')
print(f'99% Confidence Intervel: {ci99}')

90% Confidence Intervel: (np.float64(125.70003003204015), np.float64(132.48855274284193))
95% Confidence Intervel: (np.float64(125.05006509164423), np.float64(133.13851768323786))
99% Confidence Intervel: (np.float64(123.77902254153666), np.float64(134.40956023334545))


d. Use each confidence interval to evaluate $H_0:\mu=120$ months. At the matching significance levels $\alpha=.10,.05,.01$, state whether you reject or fail to reject the null hypothesis.

- $H_0:\mu=120$ months lies outside of all three of the confidence intervals, so we have evidence to reject the null at significance levels $\alpha=.10,.05,.01$. 

e. Interpret the estimate and confidence intervals in both months and years. Because living patients' eventual death times are right-censored, explain why these results do not estimate unrestricted mean lifetime.

- The estimated mean recorded time is around 129 months, which is equivalent to around 10.8 years. The bootstrap SE is around 2.1 (months) which is equivalent to around 0.18 years. The 95% confidence interval is around (125 months, 133 months), or equivalently (10.4 years, 11.1 years). All three intervals exclude 120 months (10 years), so we reject the null at every significance level, suggesting the mean recorded time to death is longer than 10 years. 

- Because living patients' eventual death times are right-censored, the results underestimate the unrestricted mean lifetime. Living patients are recorded as the time of their last follow-up, not necessarily their recorded time to death. Treating these follow-up times as if they were death times pulls the average down, so the results understate the unrestricted mean lifetime. The confidence interval describes the mean of the recorded times, not necessarily how long patients actually live.

10. In `./data/nhanes_data_17_18.csv`, select respondents who are at least 18 years old and have an observed value for `SleepHoursWeekdaysOrWorkdays`. Estimate their mean weekday sleep.
   
   a. Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of the mean. Report the seed and number of bootstrap replications.
   
   b. Compute the bootstrap standard error as the sample standard deviation of the bootstrap estimates (`ddof=1`).
   
   c. Using the original estimate and the bootstrap standard error, construct centered-Normal 90%, 95%, and 99% confidence intervals.
   
   d. Use each confidence interval to evaluate $H_0:\mu=8$ hours. At the matching significance levels $\alpha=.10,.05,.01$, state whether you reject or fail to reject the null hypothesis.
   
   e. Interpret the result and discuss whether its statistical and practical significance tell the same story. For this exercise, treat observations as independent and unweighted. Because the complex survey design is ignored, do not describe the result as nationally representative or the intervals as survey-adjusted.

In [26]:
nhanes = pd.read_csv("./data/nhanes_data_17_18.csv")
nhanes

C:\Users\corri\AppData\Local\Temp\ipykernel_14628\546380908.py:1: DtypeWarning: Columns (0: ServedActiveDutyInUsArmedForces) have mixed types. Specify dtype option on import or set low_memory=False.
  nhanes = pd.read_csv("./data/nhanes_data_17_18.csv")


,SEQN,GeneralHealthCondition,EverBreastfedOrFedBreastmilk,AgeStoppedBreastfeedingdays,AgeFirstFedFormuladays,AgeStoppedReceivingFormuladays,AgeStartedOtherFoodbeverage,AgeFirstFedMilkdays,TypeOfMilkFirstFedWholeMilk,TypeOfMilkFirstFed2Milk,...,DaysSmokedCigsDuringPast30Days,AvgCigarettesdayDuringPast30Days,TriedToQuitSmoking,TimesStoppedSmokingCigarettes,HowLongWereYouAbleToStopSmoking,UnitOfMeasureDayweekmonthyear_2_SMQ,CurrentSelfreportedHeightInches,CurrentSelfreportedWeightPounds,TriedToLoseWeightInPastYear,TimesLost10LbsOrMoreToLoseWeight
0,93703.0,NaN,1.0,273.0,1.0,365.0,152.0,365.0,Whole or regular milk,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,93704.0,NaN,1.0,60.0,3.0,365.0,126.0,365.0,Whole or regular milk,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,93705.0,Good,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,63.0,165.0,0.0,11 times or more
3,93706.0,Very good,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,68.0,145.0,0.0,Never
4,93707.0,Good,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8361,102952.0,Very good,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,117.0,0.0,Never
8362,102953.0,Fair or,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,65.0,218.0,0.0,Never
8363,102954.0,Good,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,66.0,150.0,0.0,Never
8364,102955.0,Very good,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
adult_sleep_df = nhanes[nhanes['AgeInYearsAtScreening'] >= 18]
adult_sleep_df = adult_sleep_df.dropna(subset=['SleepHoursWeekdaysOrWorkdays'])
adult_sleep_mean = adult_sleep_df['SleepHoursWeekdaysOrWorkdays'].mean()
adult_sleep_mean

np.float64(7.62124248496994)

- Estimate for mean weekly sleep is: 7.62 hours.

a. Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of the mean. Report the seed and number of bootstrap replications.

In [30]:
rng = np.random.default_rng(seed)
R = 2_500                            

replications = []
for r in range(R):
    df_r = adult_sleep_df.sample(frac = 1.0, replace = True, random_state = rng)
    mean_r = df_r['SleepHoursWeekdaysOrWorkdays'].mean()
    replications.append(mean_r)
replications = np.array(replications)

In [31]:
print(f'Seed = {seed}')
print(f'Replications = {R}')

Seed = 2134
Replications = 2500


b. Compute the bootstrap standard error as the sample standard deviation of the bootstrap estimates (`ddof=1`).

In [32]:
sleep_se = np.std(replications, ddof=1)
print(f'Bootstrapped SE = {sleep_se}')

Bootstrapped SE = 0.02286440196886463


c. Using the original estimate and the bootstrap standard error, construct centered-Normal 90%, 95%, and 99% confidence intervals.

In [34]:
ci90 = (adult_sleep_mean - 1.645*sleep_se, adult_sleep_mean + 1.645*sleep_se)
ci95 = (adult_sleep_mean - 1.960*sleep_se, adult_sleep_mean + 1.960*sleep_se)
ci99 = (adult_sleep_mean - 2.576*sleep_se, adult_sleep_mean + 2.576*sleep_se)

print(f'90% Confidence Interval: {ci90}')
print(f'95% Confidence Interval: {ci95}')
print(f'99% Confidence Interval: {ci99}')

90% Confidence Interval: (np.float64(7.583630543731158), np.float64(7.658854426208722))
95% Confidence Interval: (np.float64(7.576428257110965), np.float64(7.666056712828915))
99% Confidence Interval: (np.float64(7.562343785498145), np.float64(7.6801411844417355))


d. Use each confidence interval to evaluate $H_0:\mu=8$ hours. At the matching significance levels $\alpha=.10,.05,.01$, state whether you reject or fail to reject the null hypothesis.

- 8 hours is not included in any of the three confidence intervals, so we reject $H_0:\mu=8$ hours at all three significance levels $\alpha=.10,.05,.01$. The 90% confidence interval is (7.583630543731158, 7.658854426208722), the 95% confidence interval is (7.576428257110965, 7.666056712828915), and the 99% confidence interval is (7.562343785498145, 7.6801411844417355).

e. Interpret the result and discuss whether its statistical and practical significance tell the same story. For this exercise, treat observations as independent and unweighted. Because the complex survey design is ignored, do not describe the result as nationally representative or the intervals as survey-adjusted.

- Out of the adults included in this sample, the average weekday sleep is around 7.62 hours and the bootstrapped SE is around 0.023. Regarding statistical significance, the evidence strongly suggests that the mean is not 8 hours and is in fact lower than 8 hours. The null ($H_0:\mu=8$ hours) is rejected at every significance  level due to lying outside of all three of the constructed confidence intervals. Thinking in terms of practical significance, the difference (8-7.62=0.38) seems small. 0.38 hours would equate to around 23 minutes less of sleep a night. With a large sample size, a small difference is easy to detect. Thus, statistical and practical significance do not necessarily tell the same story. We have statistical evidence that the mean is below 8 hours, but practically, this does not necessarily mean those in the sample are lacking sleep, since the gap is relatively small.